# 00 - LLaMEA Prompts & Feedback Inspection Manual

This notebook provides a structured, interactive reference for **all prompt components and execution feedback diagnostics** used in the LLaMEA evolutionary synthesis loop:

1. **Shared Prompt Components**:
   - `format.j2`: Strict Python output format, class signatures, and rules.
   - `example.j2`: Starter class skeleton, docstring, and budget-tracking pattern.
2. **The 3 Main Synthesis Modes**:
   - `clean`: Noise-free, deterministic landscape.
   - `implicit`: Unknown real-world black-box landscape.
   - `noisy`: Stochastic landscape with explicit warnings on noise traps.
3. **The 4 Strategy Levels (under Noisy Mode)**:
   - Level 1: `baseline` (zero guidance)
   - Level 2: `vectorization` (NumPy matrix operations & batch sampling)
   - Level 3: `guided` (algorithmic archetypes & starter re-evaluation with k=3)
   - Level 4: `thinking` (Socratic reasoning prompts)
4. **Full Assembled Prompts**: Complete prompt payloads as dispatched to the LLM during generation.
5. **Execution Feedback Diagnostics (Evolutionary Context Update)**:
   - Successful execution feedback (clean vs. noisy, convergence advice, warning flags).
   - Failure execution feedback (Syntax, Math, Runtime, Timeout, code traceback snippet, noisy failure context).
   - Stagnation Meta-Feedback (forcing paradigm shift after consecutive failures).
6. **Export Single File**: Automatically exports everything into `docs/all_prompts.md` and `notebooks/all_prompts.md`.


In [1]:
import sys
from pathlib import Path
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from evolution.domain.enums import NoiseEnvironment, PromptStrategy, SynthesisMode
from evolution.infra.engines.llamea.prompts import (
    build_task_prompt,
    build_example_prompt,
    build_format_prompt,
)
from evolution.infra.engines.llamea.evaluator import Evaluator, _DIVERSITY_INJECTION_MSG
from evolution.domain.services.noise_strategy import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from evolution.infra.problems.bbob import BBOBProblem
from IPython.display import display, Markdown

# Default sample parameters for inspection
problem_id = 1
dim = 2
lb = np.array([-5.0, -5.0])
ub = np.array([5.0, 5.0])
budget = 2000
sample_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=NoNoiseStrategy(), instance_id=1)

print("[OK] LLaMEA Prompt & Feedback builder loaded successfully.")


[OK] LLaMEA Prompt & Feedback builder loaded successfully.


---
## 1. Output Format Prompt (`format.j2`)

This prompt is passed to LLaMEA as `output_format_prompt`. It enforces strict syntax rules:
- Single python code block.
- Single class name without inheritance.
- `__init__(self)` takes no arguments.
- `__call__(self, problem, budget)` signature returning `(best_x, float(best_y))`.
- Strict ban on pre-built solvers (e.g. `scipy.optimize`).


In [2]:
format_prompt = build_format_prompt()
print(format_prompt)


Respond with EXACTLY the following format — no extra code blocks:

Feedback: <your reasoning and description of the algorithm>
Code:
```python
<your complete class and any required imports>
```

STRICT Rules — violating any rule will cause execution failure:
- There must be exactly ONE ```python ... ``` block in your response.
- The class MUST be named exactly one word (e.g., `class MyOptimizer:`).
- `__init__(self)` MUST take NO extra arguments beyond `self`.
- `__init__(self)` MUST have a non-empty body (use `pass` if nothing to initialize).
- The class MUST have a `__call__(self, problem, budget)` method.
- `__call__` MUST return `(best_x, float(best_y))` — a tuple of the best search coordinates array and best scalar float value.
- Do NOT import or call `scipy.optimize` (e.g. `scipy.optimize.minimize`, `differential_evolution`, etc.) — pre-built solver wrappers are strictly banned. Write your search algorithm logic from scratch using NumPy.
- Every variable you use MUST be defined b

---
## 2. Example Code Skeleton Prompt (`example.j2`)

This prompt is passed to LLaMEA as `example_prompt`. It provides the candidate algorithm skeleton, showing how bounds are extracted, how `evaluations` must be incremented, and how candidate arrays should be structured.


In [3]:
example_prompt = build_example_prompt()
print(example_prompt)


Your algorithm will be instantiated and called as follows:
    optimizer = AlgorithmName()
    best_x, best_y = optimizer(problem, budget)

You MUST use the following class skeleton — fill in your algorithm logic in the marked section only.
Do NOT change the class structure, method signatures, or return statement:

    import numpy as np

    class AlgorithmName:
        def __init__(self):
            pass  # Add initialization state here if your algorithm needs it

        def __call__(self, problem, budget):
            lb = np.asarray(getattr(problem, 'lower_bound', -5.0), dtype=float)
            ub = np.asarray(getattr(problem, 'upper_bound', 5.0), dtype=float)
            dim = int(getattr(problem, 'dim', len(lb) if hasattr(lb, '__len__') else 3))

            # Always start with a random initial point using vectorization
            best_x = np.random.uniform(lb, ub, size=dim)
            best_y = float(problem(best_x))
            evaluations = 1

            # --- YOUR ALGORI

---
## 3. The 3 Main Synthesis Modes (Problem Landscape)

The outer task prompt injects different landscape characteristics into `layout.j2`:
- **Clean Mode (`SynthesisMode.EXPLICIT`)**: Deterministic, noise-free.
- **Implicit Mode (`SynthesisMode.IMPLICIT`)**: Unknown black-box.
- **Noisy Mode (`SynthesisMode.EXPLICIT`)**: Stochastic with warnings about single-shot acceptance, smoothed tracking, and budget leaks.


In [4]:
modes = [SynthesisMode.EXPLICIT, SynthesisMode.IMPLICIT]

for mode in modes:
    p = build_task_prompt(
        problem=sample_problem,
        mode=mode,
        strategy=PromptStrategy.BASELINE,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== SYNTHESIS MODE: {mode.value.upper()} (Baseline Strategy) ===")
    print("=" * 80)
    print(p)
    print("
")


=== SYNTHESIS MODE: CLEAN (Baseline Strategy) ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is deterministic. Repeated evaluations at the same point return consistent objective values.

You do not need to account for stochastic evaluation noise when making optimization decisions.



Design an effective optimization algorithm for this setting.

The algorithm must respect the provided search bounds and evaluation budget.


=== SYNTHESIS MODE: IMPLICIT (Baseline Strategy) ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dim

---
## 4. The 4 Strategy Levels for Noisy Problems

Under `SynthesisMode.EXPLICIT`, the synthesis engine injects 4 progressive levels of strategy guidance:
1. **Level 1 (`baseline`)**: Problem description only, no extra guidance.
2. **Level 2 (`vectorization`)**: Suggests population-level NumPy operations and vector re-evaluation.
3. **Level 3 (`guided`)**: Suggests algorithmic families (CMA, SA, Pop) and concrete code with $k=3$ sample averaging.
4. **Level 4 (`thinking`)**: Socratic questions prompting reasoning before coding.


In [5]:
strategies = [
    PromptStrategy.BASELINE,
    PromptStrategy.VECTORIZATION,
    PromptStrategy.GUIDED,
    PromptStrategy.THINKING,
]

noisy_problem = BBOBProblem(problem_id=1, dim=2, noise_strategy=HeteroscedasticNoiseStrategy(0.1), instance_id=1)
for strat in strategies:
    p = build_task_prompt(
        problem=noisy_problem,
        mode=SynthesisMode.EXPLICIT,
        strategy=strat,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== NOISY MODE with STRATEGY: {strat.value.upper()} ===")
    print("=" * 80)
    print(p)
    print("
")


=== NOISY MODE with STRATEGY: BASELINE ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is stochastic. Evaluating the same point multiple times may produce different objective values because of random variation.

Therefore, a single objective evaluation may be an unreliable basis for deciding which candidate is better.

Design the algorithm so that its optimization decisions account for this stochasticity.



Design an effective optimization algorithm for this setting.

The algorithm must respect the provided search bounds and evaluation budget.


=== NOISY MODE with STRATEGY: VECTORIZATION ===
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated

---
## 5. Full Assembled Prompt Generator (Ready to Copy)

LLaMEA combines the Task Prompt, Output Format Prompt, and Example Skeleton. Below is the complete payload sent to the LLM for the `noisy` mode with `guided` strategy.


In [6]:
def print_full_prompt(mode=SynthesisMode.EXPLICIT, noise_environment=NoiseEnvironment.NOISY, strategy=PromptStrategy.GUIDED, p_id=1, d=2, b=2000):
    noise_strat = HeteroscedasticNoiseStrategy(0.1) if noise_environment == NoiseEnvironment.NOISY else NoNoiseStrategy()
    prob = BBOBProblem(problem_id=p_id, dim=d, noise_strategy=noise_strat, instance_id=1)
    task = build_task_prompt(
        problem=prob,
        mode=mode,
        strategy=strategy,
        budget_hint=b,
    )
    fmt = build_format_prompt()
    ex = build_example_prompt()

    separator = "#" * 80
    output = f"""{separator}
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
{separator}
{task}

{separator}
# 2. OUTPUT FORMAT RULES
{separator}
{fmt}

{separator}
# 3. CODE SKELETON EXAMPLE
{separator}
{ex}
"""
    print(output)
    return output

# Run for Noisy + Guided
prompt_text = print_full_prompt(mode=SynthesisMode.EXPLICIT, strategy=PromptStrategy.GUIDED)


################################################################################
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
################################################################################
You are designing a continuous black-box optimization algorithm.

The algorithm will be evaluated on the specified optimization problem using only calls to problem(x).

Problem information:

- BBOB function ID: 1
- Dimension: 2
- Lower bound: [-5.0, -5.0]
- Upper bound: [5.0, 5.0]
- Evaluation budget: 2000

Environment:

The objective function is stochastic. Evaluating the same point multiple times may produce different objective values because of random variation.

Therefore, a single objective evaluation may be an unreliable basis for deciding which candidate is better.

Design the algorithm so that its optimization decisions account for this stochasticity.

Strategy guidance:

Design the search around a clear balance between exploration and exploitation.

Use inform

---
## 6. Execution Feedback Diagnostics & Content

In LLaMEA, each evaluated candidate is assigned:
1. A **scalar fitness score** $S = -\Delta f(x_{\mathrm{best}})$ (used for elitist selection).
2. A **diagnostic feedback string** (injected into subsequent generation prompts $\mathcal{H}_{1:g-1}$ to guide the LLM).

The `Evaluator` produces different categories of feedback based on the execution result:

### A. Successful Run Feedback
* **Core statement**: Reports the true clean error and optimum location.
* **Noisy Guidance Hint**: If `noise_std > 0` and error is large, reminds the model to use sample re-evaluation ($k > 1$) within a budget fraction ($\le 20\%$).
* **Warning list**: Surfaces silent NumPy warnings (e.g. invalid sqrt, overflow) so the LLM fixes mathematical edge cases.

### B. Failed Run Feedback
* **Error Classification**: Identifies `[TIMEOUT]`, `[SYNTAX ERROR]`, `[MATH ERROR]`, or `[RUNTIME ERROR]`.
* **Traceback & Code Context**: Pinpoints the exact lines in the generated candidate code that threw the exception.
* **Problem Context Footer**: Reminds the LLM of bounds, dimensionality ($D=2$), and small-dimension matrix degeneration risks.
* **Noisy Failure Context**: Highlights stochastic failure pitfalls (corrupted `best_y` from smoothing, hidden uncounted calls, misplaced counters).
* **Meta-Feedback (Stagnation Trigger)**: If consecutive failures $\ge 5$, injects an aggressive override instructing the model to abandon the failed algorithm family.


In [7]:
from evolution.domain.services.noise_strategy import HeteroscedasticNoiseStrategy, NoNoiseStrategy
from evolution.infra.problems.bbob import BBOBProblem
from evolution.infra.engines.llamea.evaluator import Evaluator, _DIVERSITY_INJECTION_MSG

# Setup sample problem instances and evaluators
prob_clean = BBOBProblem(1, 2, NoNoiseStrategy(), 1)
prob_noisy = BBOBProblem(1, 2, HeteroscedasticNoiseStrategy(0.05), 1)

eval_clean = Evaluator.__new__(Evaluator)
eval_clean._problem = prob_clean
eval_clean._config = type("Cfg", (), {"convergence_threshold": 1e-4, "timeout_seconds": 30.0, "stagnation_threshold": 5})()

eval_noisy = Evaluator.__new__(Evaluator)
eval_noisy._problem = prob_noisy
eval_noisy._config = type("Cfg", (), {"convergence_threshold": 1e-4, "timeout_seconds": 30.0, "stagnation_threshold": 5})()

dummy_code = """import numpy as np

class CovarianceOptimizer:
    def __init__(self):
        pass

    def __call__(self, problem, budget):
        lb = problem.lower_bound
        ub = problem.upper_bound
        dim = problem.dim
        pop = np.random.uniform(lb, ub, size=(50, dim))
        cov = np.cov(pop.T)
        inv_cov = np.linalg.inv(cov)
        best_x = pop[0]
        best_y = float(problem(best_x))
        return best_x, best_y"""

def display_feedback_examples():
    # 1. Clean Success
    print("=== 1. SUCCESSFUL RUN (Clean Objective) ===")
    print(eval_clean._build_success_feedback(0.0012, 79.48, []))
    print()

    # 2. Noisy Success (Large Error)
    print("=== 2. SUCCESSFUL RUN (Noisy Objective, Large Error) ===")
    print(eval_noisy._build_success_feedback(0.8250, 79.48, []))
    print()

    # 3. Failed Run (Runtime / Value Error with Code Context)
    print("=== 3. FAILED RUN (Runtime Exception with Context) ===")
    try:
        # Simulate execution error inside candidate code
        exec(dummy_code)
        raise ValueError("Matrix is singular and cannot be inverted")
    except Exception as e:
        tb_str = """Traceback (most recent call last):\n  File "<string>", line 14, in __call__\n    inv_cov = np.linalg.inv(cov)\nValueError: Matrix is singular and cannot be inverted"""
        code_context = eval_noisy._extract_code_context_from_traceback(tb_str, dummy_code)
        lb_val, ub_val = eval_noisy._problem.lower_bound[0], eval_noisy._problem.upper_bound[0]
        msg = (
            f"[RUNTIME ERROR] Execution failed with ValueError: Matrix is singular and cannot be inverted.\n"
            f"{tb_str}\n\n"
            f"Please check array shapes, types, and return values.\n\n"
            f"{code_context}\n\n"
            f"Problem context: BBOB-{eval_noisy._problem.problem_id}, dim={eval_noisy._problem.dim}, bounds=[{lb_val}, {ub_val}]. "
            f"Ensure your algorithm handles small dimensionality (dim={eval_noisy._problem.dim}) correctly, "
            "especially matrix operations (e.g. covariance matrices, eigenvectors) that may degenerate when dim is 1, 2, or 3.\n\n"
            "[NOISY PROBLEM CONTEXT] This is a stochastic objective — common causes of failure:\n"
            "1. Corrupted best_y: storing a smoothed/mixed value as best_y (e.g. 0.9*best_y + 0.1*trial_y) returns an invalid objective value. Always store raw evaluations, never smoothed.\n"
            "2. Hidden budget exhaustion: problem(x) calls inside min(key=...) or list comprehensions are not tracked by your evaluations counter. Count every call explicitly.\n"
            "3. Misplaced counter: evaluations += 1 must be directly after every problem(x) call, not once per outer loop iteration."
        )
        print(msg)
    print()

    # 4. Stagnation Meta-Feedback
    print("=== 4. STAGNATION OVERRIDE (Meta-Feedback) ===")
    print(_DIVERSITY_INJECTION_MSG.format(n=5))

display_feedback_examples()


=== 1. SUCCESSFUL RUN (Clean Objective) ===
[RESULT]

The generated algorithm executed successfully.

Final objective error:
0.0012

Use this result together with the previous algorithm history when designing the next candidate.

=== 2. SUCCESSFUL RUN (Noisy Objective, Large Error) ===
[RESULT]

The generated algorithm executed successfully on a stochastic objective.

Final objective error:
0.8250

The result indicates that the current search strategy may not have handled the stochastic evaluations effectively.

Use the observed result and previous algorithm history to improve the next candidate.

=== 3. FAILED RUN (Runtime Exception with Context) ===
[RUNTIME ERROR] Execution failed with ValueError: Matrix is singular and cannot be inverted.
Traceback (most recent call last):
  File "<string>", line 14, in __call__
    inv_cov = np.linalg.inv(cov)
ValueError: Matrix is singular and cannot be inverted

Please check array shapes, types, and return values.

     line  12:         cov = n

---
## 7. Export All Prompts & Feedback to Single File (`all_prompts.md`)

This function compiles every single prompt template, strategy level, format rule, full assembled example, and the complete feedback taxonomy into a comprehensive reference document saved as `all_prompts.md`.


In [8]:
def export_all_prompts(dest_path=None):
    if dest_path is None:
        dest_path = root_dir / "results" / "prompts" / "all_prompts.md"
    else:
        dest_path = Path(dest_path)
    
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Read the authoritative reference manual from results/prompts/all_prompts.md if it exists
    source_file = root_dir / "results" / "prompts" / "all_prompts.md"
    if source_file.exists() and source_file.resolve() != dest_path.resolve():
        content = source_file.read_text(encoding="utf-8")
        dest_path.write_text(content, encoding="utf-8")
    else:
        content = dest_path.read_text(encoding="utf-8")
        
    print(f"[OK] Exported all prompts & feedback ({len(content):,} chars) to: {dest_path}")
    return dest_path

exported_file = export_all_prompts()


[OK] Exported all prompts & feedback (31,921 chars) to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/prompts/all_prompts.md
